# Part 3: LoRA Fine-Tuning for Nemotron-3-Nano-30B

Fine-tune using the CoT training data generated in Part 2.

## Model Architecture: Nemotron-H (Hybrid Transformer + Mamba-2)

**This is NOT a standard transformer.** Nemotron-3-Nano-30B uses `NemotronHForCausalLM`:
- **52 total layers**: 23 Attention (M) + 23 Mamba-2 (E) + 6 MoE (*)
- **30B params, 3.5B active** (sparse MoE with 128 routed experts, 6 active)
- **Layer pattern:** `MEMEM*EMEMEM*EMEMEM*EMEMEM*EMEMEM*EMEMEMEM*EMEMEMEME`
- **Chat template:** ChatML with `<think>` reasoning tags

## Critical Technical Requirements
- LoRA rank ≤ 32, must include `adapter_config.json`
- Target modules must cover **BOTH** Transformer attention AND Mamba-2 layers
- Chat template must match Nemotron's ChatML format exactly
- All training sequences must fit within `max_model_len=8192`

**Hardware:** GPU with ≥ 24GB VRAM (Google Cloud G4 VM with RTX PRO 6000)

## 0. Install Dependencies

Run in terminal or uncomment below:
```bash
pip install unsloth
pip install --upgrade transformers trl peft accelerate bitsandbytes
# For Mamba-2 support (hybrid architecture):
pip install mamba-ssm causal-conv1d
```

In [1]:
import subprocess, sys, os
from pathlib import Path
def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read()
            rel_pack_path = (pth_file.parent/relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))



offline_dir = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages"
Nvidia_whl_dir = "/kaggle/input/datasets/manish756/nvidia-wheel/packages"
target_dir = "/kaggle/working/packages"

os.makedirs(target_dir, exist_ok=True)
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", Nvidia_whl_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
    print("Installed from offline packages")
# Add to Python path
sys.path.append(target_dir)
resolve_python_path(target_dir)

import datasets

append /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
torchvision 0.26.0.dev20260324+cu128 requires torch==2.12.0.dev20260324, but you have torch 2.11.0 which is incompatible.
cuda-python 12.9.6 requires cuda-bindings~=12.9.6, but you have cuda-bindings 13.2.0 which is incompatible.
ydata-profiling 4.18.1 requires numpy<2.4,>=1.22, but you have numpy 2.4.4 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatibl

Installed from offline packages


In [3]:
!pip install  --no-index /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/flash_attn-2.8.3+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
flash-attn is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [ ]:
# unsloth_dir = '/kaggle/input/datasets/victorogobi/unsloth-latest/unsloth_wheels'
# if os.path.exists(unsloth_dir):
#     subprocess.check_call([
#         sys.executable, "-m", "pip", "install", "-q",
#         "--no-index",
#         "--find-links", offline_dir,
#         "--target", target_dir,
#         "datasets", "trl"
#     ])
#     print("Installed from offline packages")

In [4]:
import json
import os
import torch
from datasets import Dataset
for candidate in ['/kaggle/input/datasets/manish756/nemotron-dataset/train_cot_v3_metric_aligned.jsonl', 'train_cot2.0.jsonl', 'train_cot.jsonl']:
    if os.path.exists(candidate):
        JSONL_FILE = candidate
        break

data = []
with open(JSONL_FILE, 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Loaded {len(data)} training examples from {JSONL_FILE}")
print(f"Message roles: {[m['role'] for m in data[0]['messages']]}")

# Verify: no system prompt in v3 data
has_system = any(m['role'] == 'system' for m in data[0]['messages'])
print(f"Has system prompt: {has_system} {'(CORRECT - matches eval)' if not has_system else '(WARNING - eval has no system prompt!)'}")

# Verify: <think> tags present
assistant_msg = data[0]['messages'][-1]['content']
has_think = '<think>' in assistant_msg and '</think>' in assistant_msg
print(f"Has <think> tags: {has_think}")

# Verify: \boxed{} present
has_boxed = '\\boxed{' in assistant_msg
print(f"Has \\boxed{{}}: {has_boxed}")

print(f"\n--- Sample Assistant Response (first 300 chars) ---")
print(assistant_msg[:300])

Loaded 9500 training examples from /kaggle/input/datasets/manish756/nemotron-dataset/train_cot_v3_metric_aligned.jsonl
Message roles: ['user', 'assistant']
Has system prompt: False (CORRECT - matches eval)
Has <think> tags: True
Has \boxed{}: True

--- Sample Assistant Response (first 300 chars) ---
<think>
I need to find the transformation rule from these input→output examples and apply it to 00110100.

Let me analyze the examples to identify the pattern:

01010001 → 11011101
00001001 → 01101101
00010101 → 01010101
11111111 → 10000001

Studying the bit-by-bit transformation pattern across all 


## 1. Load Model with Unsloth

In [5]:
from unsloth import FastLanguageModel

# ==========================================================================
# FIX 1: Correct HuggingFace Model ID
# The competition uses Nemotron-3-Nano-30B which is a HYBRID model:
#   - Architecture: NemotronHForCausalLM (model_type: nemotron_h)
#   - 52 layers: 23 Attention + 23 Mamba-2 + 6 MoE
#   - 30B total params, 3.5B active (sparse MoE)
# ==========================================================================
MODEL_NAME = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
MAX_SEQ_LENGTH = 8192  # Match competition's max_model_len
LORA_RANK = 32  # Maximum allowed by competition

# Load model with 4-bit quantization for memory efficiency
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect (bf16 preferred if supported)
    load_in_4bit=True,  # QLoRA for memory efficiency
)

print(f"Model loaded: {MODEL_NAME}")
print(f"Model type: {model.config.model_type}")
print(f"Model dtype: {model.dtype}")
print(f"Vocab size: {model.config.vocab_size}")

ModuleNotFoundError: No module named 'unsloth'

## 2. Add LoRA Adapters

**Critical architecture note:** Nemotron-3-Nano-30B is a **hybrid** model with:
- **23 Transformer Attention layers** → `q_proj`, `k_proj`, `v_proj`, `o_proj`
- **23 Mamba-2 SSM layers** → `in_proj`, `out_proj`
- **MLP/MoE layers** → `up_proj`, `down_proj`

You MUST target both Transformer and Mamba-2 modules. The old config only targeted
Transformer layers + a nonexistent `gate_proj`, leaving 23 Mamba-2 layers completely
unadapted. This is the single biggest fix for training quality.

In [ ]:
# ==========================================================================
# FIX 2: Correct LoRA Target Modules for Hybrid Architecture
#
# OLD (WRONG):  ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
#   - "gate_proj" does NOT EXIST in Nemotron (it uses squared ReLU, not gated MLP)
#   - Missing Mamba-2 layers entirely (23 layers unadapted!)
#
# NEW (CORRECT): All Attention + Mamba-2 + MLP modules
# ==========================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,  # LoRA rank = 32 (max allowed by competition)
    target_modules=[
        # Transformer Attention layers (23 layers)
        "q_proj", "k_proj", "v_proj", "o_proj",
        # Mamba-2 SSM layers (23 layers) — CRITICAL ADDITION
        "in_proj", "out_proj",
        # MLP / MoE expert layers
        "up_proj", "down_proj",
    ],
    lora_alpha=64,  # Typically 2x rank for good scaling
    lora_dropout=0,  # Unsloth optimized — 0 is faster and works well
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory optimization
    random_state=42,
)

# Print trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

# Verify the target modules were applied correctly
from peft import PeftModel
if hasattr(model, 'peft_config'):
    config = list(model.peft_config.values())[0]
    print(f"\nLoRA config:")
    print(f"  Rank: {config.r}")
    print(f"  Alpha: {config.lora_alpha}")
    print(f"  Target modules: {config.target_modules}")
    print(f"  Dropout: {config.lora_dropout}")

## 3. Prepare Dataset

### Format Alignment with Evaluation

The metric code shows exactly how the model is prompted during evaluation:
```python
# From generate_predictions() in the metric:
user_content = item.prompt + '\nPlease put your final answer inside `\\boxed{}`...'
prompt = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': user_content}],  # NO system prompt
    enable_thinking=True,                           # <think> tags enabled
)
```

Our v3 training data already matches this format:
- No system prompt
- Boxed instruction appended to user message  
- `<think>reasoning</think>\boxed{answer}` in assistant response

The tokenizer's `apply_chat_template` will format this as:
```
<|im_start|>user
{puzzle + boxed instruction}<|im_end|>
<|im_start|>assistant
<think>
{step-by-step reasoning}
</think>
\boxed{answer}<|im_end|>
```

In [ ]:
# ==========================================================================
# Apply Tokenizer Chat Template
#
# The v3 data already has <think> tags in the assistant content.
# We just need to apply the tokenizer's chat template to get the
# correct special tokens (<|im_start|>, <|im_end|>, etc.)
# ==========================================================================

dataset = Dataset.from_list(data)

def format_example(example):
    messages = example['messages']
    
    # The v3 data already has the correct format:
    #   user: puzzle + boxed instruction
    #   assistant: <think>reasoning</think>\boxed{answer}
    # 
    # If data has a system prompt (v1/v2), skip it to match evaluation
    formatted_messages = [m for m in messages if m['role'] != 'system']
    
    # If assistant content doesn't have <think> tags yet (v1/v2 data), add them
    assistant_msg = formatted_messages[-1]
    if '<think>' not in assistant_msg['content']:
        import re
        content = assistant_msg['content']
        boxed_match = re.search(r'(\\boxed\{.*?\})\s*$', content)
        if boxed_match:
            reasoning = content[:boxed_match.start()].strip()
            boxed_answer = boxed_match.group(1)
            formatted_messages[-1] = {
                'role': 'assistant',
                'content': f"<think>\n{reasoning}\n</think>\n{boxed_answer}"
            }
    
    # Apply tokenizer's chat template
    try:
        text = tokenizer.apply_chat_template(
            formatted_messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception as e:
        # Manual ChatML fallback
        parts = []
        for m in formatted_messages:
            parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
        text = '\n'.join(parts)
    
    return {'text': text}

dataset = dataset.map(format_example)

# === Token Length Validation ===
print("=== Token Length Validation ===\n")

sample_tokens = tokenizer(dataset[0]['text'], return_tensors='pt')
print(f"Sample token length: {sample_tokens['input_ids'].shape[1]}")

# Check ALL examples
print("Checking all examples against max_model_len=8192...")
lengths = []
for i in range(len(dataset)):
    tokens = tokenizer(dataset[i]['text'], return_tensors='pt')
    lengths.append(tokens['input_ids'].shape[1])

import numpy as np
lengths = np.array(lengths)
print(f"  Min:    {lengths.min()} tokens")
print(f"  Max:    {lengths.max()} tokens")
print(f"  Mean:   {lengths.mean():.0f} tokens")
print(f"  Median: {np.median(lengths):.0f} tokens")
print(f"  P95:    {np.percentile(lengths, 95):.0f} tokens")
print(f"  P99:    {np.percentile(lengths, 99):.0f} tokens")

over_limit = (lengths > MAX_SEQ_LENGTH).sum()
if over_limit > 0:
    print(f"\n  WARNING: {over_limit} examples exceed {MAX_SEQ_LENGTH} tokens!")
else:
    print(f"\n  All {len(lengths)} examples fit within {MAX_SEQ_LENGTH} token limit.")

# Show the formatted template
print(f"\n{'='*70}")
print("Formatted template (first 600 chars):")
print("="*70)
print(dataset[0]['text'][:600])

## 4. Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=4,
    packing=True,  # Pack multiple short examples into one sequence
    args=TrainingArguments(
        output_dir="./outputs",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,  # Effective batch size = 16
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_steps=100,
        save_total_limit=3,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Training loss: {trainer_stats.training_loss:.4f}")

## 5. Save LoRA Adapter

In [ ]:
import os
import zipfile

# Save LoRA adapter
ADAPTER_DIR = "./lora_adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"Adapter saved to {ADAPTER_DIR}")
print(f"Files:")
for f in os.listdir(ADAPTER_DIR):
    size = os.path.getsize(os.path.join(ADAPTER_DIR, f))
    print(f"  {f}: {size/1024/1024:.2f} MB")

# Verify adapter_config.json exists (required by competition)
assert os.path.exists(os.path.join(ADAPTER_DIR, 'adapter_config.json')), \
    "adapter_config.json is missing! This is required for submission."
print("\n✓ adapter_config.json exists")

## 6. Create Submission

In [ ]:
# Package as submission.zip
SUBMISSION_PATH = "submission.zip"

with zipfile.ZipFile(SUBMISSION_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(ADAPTER_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, ADAPTER_DIR)
            zf.write(file_path, arcname)

zip_size = os.path.getsize(SUBMISSION_PATH)
print(f"\nSubmission created: {SUBMISSION_PATH} ({zip_size/1024/1024:.2f} MB)")
print(f"\nFiles in submission.zip:")
with zipfile.ZipFile(SUBMISSION_PATH, 'r') as zf:
    for info in zf.infolist():
        print(f"  {info.filename}: {info.file_size/1024:.1f} KB")

## 7. Quick Local Evaluation (Optional)

Test the adapter locally with vLLM to estimate leaderboard score before submitting.

In [ ]:
# ==========================================================================
# Local Evaluation — MATCHES THE EXACT COMPETITION METRIC CODE
# (pip install vllm)
#
# This replicates generate_predictions() and verify() from the metric.
# ==========================================================================

import re
import math

def extract_final_answer(text):
    """Exact copy of the competition's answer extraction logic."""
    if text is None:
        return 'NOT_FOUND'
    # Search for boxed answer
    matches = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()
    # Fallback patterns
    for pattern in [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()
    # Last number
    matches = re.findall(r'-?\d+(?:\.\d+)?', text)
    if matches:
        return matches[-1]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'


def verify(stored_answer, predicted):
    """Exact copy of the competition's verification logic."""
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()
    # Binary strings: exact match
    if re.fullmatch(r'[01]+', stored_answer):
        return predicted.lower() == stored_answer.lower()
    try:
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


# --- Uncomment below to run local evaluation with vLLM ---

# from vllm import LLM, SamplingParams
# from vllm.lora.request import LoRARequest
# import pandas as pd
# 
# EVAL_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'
# 
# llm = LLM(
#     model=str(MODEL_NAME),
#     tensor_parallel_size=1,
#     max_num_seqs=64,
#     gpu_memory_utilization=0.85,
#     dtype='auto',
#     max_model_len=8192,
#     trust_remote_code=True,
#     enable_lora=True,
#     max_lora_rank=32,
#     enable_prefix_caching=True,
#     enable_chunked_prefill=True,
# )
# 
# sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=7680)
# eval_tokenizer = llm.get_tokenizer()
# 
# # Test on training data sample
# test_df = pd.read_csv('train.csv').sample(50, random_state=42)
# 
# prompts = []
# for _, row in test_df.iterrows():
#     user_content = row['prompt'] + EVAL_SUFFIX
#     prompt = eval_tokenizer.apply_chat_template(
#         [{'role': 'user', 'content': user_content}],
#         tokenize=False,
#         add_generation_prompt=True,
#         enable_thinking=True,
#     )
#     prompts.append(prompt)
# 
# lora_request = LoRARequest('adapter', 1, ADAPTER_DIR)
# outputs = llm.generate(prompts, sampling_params, lora_request=lora_request)
# 
# correct = 0
# for (_, row), output in zip(test_df.iterrows(), outputs):
#     raw_text = output.outputs[0].text
#     predicted = extract_final_answer(raw_text)
#     match = verify(str(row['answer']), predicted)
#     correct += match
#     if not match:
#         print(f"WRONG: expected={row['answer']}, got={predicted}")
# 
# print(f"\nLocal accuracy: {correct}/{len(test_df)} ({correct/len(test_df)*100:.1f}%)")

print("Local evaluation functions defined (extract_final_answer, verify).")
print("Uncomment the vLLM section above to run full local evaluation.")

## Summary

### Fixes Applied (from metric code analysis):

| Issue | Old (Wrong) | New (Correct) |
|---|---|---|
| **Model ID** | `nvidia/Nemotron-3-Nano-30B` | `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16` |
| **LoRA targets** | Only Attention + nonexistent `gate_proj` | Attention + **Mamba-2** (`in_proj`, `out_proj`) + MLP |
| **System prompt** | Included in training | **Removed** (evaluation sends no system prompt) |
| **User prompt** | Raw puzzle text | Puzzle + `\nPlease put your final answer inside \boxed{}...` |
| **Assistant format** | Plain CoT + `\boxed{}` | `<think>` reasoning `</think>` + `\boxed{}` |
| **Local eval** | Generic vLLM setup | **Exact replica** of competition metric code |

### Verification checklist:
- [ ] `adapter_config.json` exists in output
- [ ] LoRA rank ≤ 32
- [ ] All training sequences < 8192 tokens
- [ ] No system prompt in training data
- [ ] `<think>` tags present in assistant responses
- [ ] `\boxed{}` at end of every response
- [ ] Local evaluation uses same `extract_final_answer()` and `verify()` as competition

### To improve further:
- **More data**: Generate synthetic puzzles + CoT (target 20K-50K examples)
- **Better CoT**: Use Claude/GPT-4 for higher-quality reasoning chains
- **RL fine-tuning**: GRPO on top of SFT using programmatic verifiers
- **Hyperparameter sweep**: Learning rate (1e-5 to 5e-5), epochs (2-5), alpha (32-128)
- **Category-specific**: Oversample weak categories, generate targeted synthetic data